# Clean vectors → interim

Normalize VectorNet detections onto modelling keys (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/01_raw/vector/` + `countries.csv`, `subnational_locations.csv`, `vector_taxon_groups.csv` |
| Output | `data/02_interim/vectors_normalized.parquet` |
| H3 | `H3_RESOLUTION` from `vector_guard.resolve` (locked = 4) |
| Time | Midpoint year-month of `eventDate` intervals (start if only one bound) |
| Coords | Path A GPS preferred; Path B centroid via `higherGeographyID` / NUTS when GPS missing |

Rows without usable year-month or `vector_group` are kept for QA but flagged; merge will drop them from panel keys.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_interim_dir,
    find_repo_root,
    load_vector_detections,
)
from vector_guard.resolve import (
    GEO_PRECISIONS,
    H3_RESOLUTION,
    assign_h3,
    build_subnational_centroid_index,
    build_valid_iso2_set,
    load_countries,
    load_subnational_locations,
    load_vector_taxon_groups,
    parse_year_month_vectornet,
    resolve_coords_ladder,
    resolve_vector_country_iso2,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = INTERIM_DIR / "vectors_normalized.parquet"

assert H3_RESOLUTION is not None and H3_RESOLUTION >= 0
REPO_ROOT, OUT_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/02_interim/vectors_normalized.parquet'),
 4)

## Load raw + reference

In [2]:
raw = load_vector_detections()
countries = load_countries()
subnational = load_subnational_locations()
taxon_groups = load_vector_taxon_groups()

valid_iso2 = build_valid_iso2_set(countries)
centroid_index = build_subnational_centroid_index(subnational)
taxon_map = {
    str(row.scientific_name): str(row.vector_group)
    for row in taxon_groups.itertuples(index=False)
}

{
    "raw_rows": len(raw),
    "countries": len(countries),
    "subnational": len(subnational),
    "taxon_groups": len(taxon_groups),
}

{'raw_rows': 120158,
 'countries': 251,
 'subnational': 2472,
 'taxon_groups': 224}

## Resolve country, ecology, time, coords, H3

- **country_iso2** — prefer published `countryCode` when present; else first two chars of `higherGeographyID`; UK/EL → GB/GR; validate against `countries.csv`
- **vector_group** — join `scientificName` → `vector_taxon_groups.csv`
- **year / month** — midpoint of `eventDate` start/end (documented rule)
- **geo_precision / lat / lon** — coord ladder with Path B allowed via NUTS / `higherGeographyID`
- **h3_cell** — assigned at `H3_RESOLUTION` when coords resolve

In [3]:
def _text(value: object) -> str:
    text = str(value or "").strip()
    if text.lower() in {"nan", "none", "null", "na", "n/a", "<na>"}:
        return ""
    return text


records = raw.to_dict(orient="records")

country_iso2 = [
    resolve_vector_country_iso2(
        country_code=row.get("countryCode", ""),
        higher_geography_id=row.get("higherGeographyID", ""),
        valid_iso2=valid_iso2,
    )
    for row in records
]

scientific_name = [_text(row.get("scientificName", "")) for row in records]
vector_group = [taxon_map.get(name, "") for name in scientific_name]

year_month = [parse_year_month_vectornet(row.get("eventDate", "")) for row in records]
year = [pair[0] if pair else pd.NA for pair in year_month]
month = [pair[1] if pair else pd.NA for pair in year_month]

nuts_id = [_text(row.get("higherGeographyID", "")) for row in records]

coords = [
    resolve_coords_ladder(
        lat=row.get("decimalLatitude", ""),
        lon=row.get("decimalLongitude", ""),
        coord_source="",
        country_iso2=iso2,
        nuts_id=nuts,
        centroid_index=centroid_index,
        allow_centroid=True,
    )
    for row, iso2, nuts in zip(records, country_iso2, nuts_id, strict=True)
]
lat = [item[0] for item in coords]
lon = [item[1] for item in coords]
geo_precision = [item[2] for item in coords]
h3_cell = [assign_h3(la, lo, H3_RESOLUTION) for la, lo in zip(lat, lon, strict=True)]

out = pd.DataFrame(
    {
        "record_id": [_text(row.get("id", "") or row.get("occurrenceID", "")) for row in records],
        "scientific_name": scientific_name,
        "vector_group": vector_group,
        "country_iso2": country_iso2,
        "lat": lat,
        "lon": lon,
        "geo_precision": geo_precision,
        "h3_cell": h3_cell,
        "year": year,
        "month": month,
        "nuts_id": nuts_id,
        "individual_count": [_text(row.get("individualCount", "")) for row in records],
        "sampling_protocol": [_text(row.get("samplingProtocol", "")) for row in records],
        "sample_size_value": [_text(row.get("sampleSizeValue", "")) for row in records],
    }
)

# Nullable ints for panel keys
out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
out["month"] = pd.to_numeric(out["month"], errors="coerce").astype("Int64")

out.head()

,record_id,scientific_name,vector_group,country_iso2,lat,lon,geo_precision,h3_cell,year,month,nuts_id,individual_count,sampling_protocol,sample_size_value
0,1141be2c-6c03-4685-922b-d7e73d77bd0d,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,0,Collection from live/dead hosts,21
1,3768ff90-473f-450d-9af1-3ec808691f17,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,0,Collection from live/dead hosts,21
2,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,4,Collection from live/dead hosts,21
3,88d2c677-9895-4d5d-9311-ac6491c175e1,Ixodes ricinus,tick,RO,46.7600,23.5600,exact_coords,841e0b3ffffffff,2018,7,RO113,4,Collection from live/dead hosts,
4,42e04dd7-fac1-43dd-a8dc-2460f669302a,Ixodes ricinus,tick,EE,58.7365,25.6682,exact_coords,8411341ffffffff,2013,7,EE009,75,Collection from live/dead hosts,


## QA coverage

In [4]:
n = len(out)
qa = {
    "rows": n,
    "iso2_mapped_share": float((out["country_iso2"] != "").mean()),
    "vector_group_mapped_share": float((out["vector_group"] != "").mean()),
    "year_month_share": float((out["year"].notna() & out["month"].notna()).mean()),
    "h3_assigned_share": float(out["h3_cell"].notna().mean()),
    "panel_key_ready_share": float(
        (
            (out["vector_group"] != "")
            & out["year"].notna()
            & out["month"].notna()
            & ((out["h3_cell"].notna()) | (out["country_iso2"] != ""))
        ).mean()
    ),
}
display(pd.Series(qa))

print("geo_precision")
display(out["geo_precision"].value_counts(dropna=False))
print("vector_group")
display(out["vector_group"].value_counts(dropna=False))

unmapped_taxa = (
    out.loc[out["vector_group"] == "", "scientific_name"]
    .value_counts()
    .rename_axis("scientific_name")
    .reset_index(name="rows")
)
print(f"Unmapped taxa: {len(unmapped_taxa)}")
display(unmapped_taxa.head(15))

print("year range")
display(out.loc[out["year"].notna(), "year"].agg(["min", "max", "median"]))

rows                         120158.000000
iso2_mapped_share                 0.999983
vector_group_mapped_share         1.000000
year_month_share                  0.991736
h3_assigned_share                 1.000000
panel_key_ready_share             0.991736
dtype: float64

geo_precision


geo_precision
exact_coords    120158
Name: count, dtype: int64

vector_group


vector_group
biting_midge    48622
mosquito        48094
sandfly         18927
tick             4515
Name: count, dtype: int64

Unmapped taxa: 0


,scientific_name,rows


year range


min       1957.0
max       2025.0
median    2017.0
Name: year, dtype: float64

## Assert + write parquet

In [5]:
required_cols = [
    "record_id",
    "scientific_name",
    "vector_group",
    "country_iso2",
    "lat",
    "lon",
    "geo_precision",
    "h3_cell",
    "year",
    "month",
    "nuts_id",
    "individual_count",
    "sampling_protocol",
    "sample_size_value",
]
assert list(out.columns) == required_cols
assert out["geo_precision"].isin(GEO_PRECISIONS).all()

has_coords = out["lat"].notna() & out["lon"].notna()
assert out.loc[has_coords, "lat"].between(-90, 90).all()
assert out.loc[has_coords, "lon"].between(-180, 180).all()
assert (
    out.loc[has_coords, "geo_precision"]
    .isin({"exact_coords", "subnational_centroid"})
    .all()
)
assert out.loc[has_coords, "h3_cell"].notna().all()
assert out.loc[~has_coords, "h3_cell"].isna().all()
assert (out.loc[out["month"].notna(), "month"].between(1, 12)).all()
assert out["record_id"].ne("").all()

out.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
display(out.head(10))
display(out.tail(5))

Wrote /workspaces/vector-guard/data/02_interim/vectors_normalized.parquet (120,158 rows)


,record_id,scientific_name,vector_group,country_iso2,lat,lon,geo_precision,h3_cell,year,month,nuts_id,individual_count,sampling_protocol,sample_size_value
0,1141be2c-6c03-4685-922b-d7e73d77bd0d,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,0,Collection from live/dead hosts,21
1,3768ff90-473f-450d-9af1-3ec808691f17,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,0,Collection from live/dead hosts,21
2,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,Ixodes ricinus,tick,SE,56.2000,16.4010,exact_coords,841f283ffffffff,2009,7,SE213,4,Collection from live/dead hosts,21
3,88d2c677-9895-4d5d-9311-ac6491c175e1,Ixodes ricinus,tick,RO,46.7600,23.5600,exact_coords,841e0b3ffffffff,2018,7,RO113,4,Collection from live/dead hosts,
4,42e04dd7-fac1-43dd-a8dc-2460f669302a,Ixodes ricinus,tick,EE,58.7365,25.6682,exact_coords,8411341ffffffff,2013,7,EE009,75,Collection from live/dead hosts,
5,94ec7b26-cf79-4496-894d-101a216d9952,Ixodes ricinus,tick,RO,46.7600,23.5000,exact_coords,841e0b3ffffffff,2018,7,RO113,5,Collection from live/dead hosts,
6,a05cc859-1b03-4d95-8975-ced548d20501,Ixodes ricinus,tick,EE,59.2260,26.1335,exact_coords,841135dffffffff,2013,7,EE009,83,Collection from live/dead hosts,
7,73b7b9a4-08d2-40f3-9d73-4e4d034981e1,Ixodes ricinus,tick,EE,58.5075,22.4107,exact_coords,84089e5ffffffff,2013,7,EE004,140,Collection from live/dead hosts,
8,115abf54-d72f-4d32-bded-a948f4a72b26,Ixodes ricinus,tick,EE,58.2493,27.3023,exact_coords,8411363ffffffff,2014,1,EE008,32,Collection from live/dead hosts,
9,f578f41b-6fbf-4210-b8e3-15a73e712632,Ixodes ricinus,tick,RO,46.7600,23.5800,exact_coords,841e0b3ffffffff,2018,7,RO113,5,Collection from live/dead hosts,


,record_id,scientific_name,vector_group,country_iso2,lat,lon,geo_precision,h3_cell,year,month,nuts_id,individual_count,sampling_protocol,sample_size_value
120153,459b04a4-3112-4ee7-9f73-ae3e1fca47b9,Aedes geniculatus,mosquito,CH,45.8497,8.98612,exact_coords,841f9d7ffffffff,2018,6,CH070,38,Ovitrap,14
120154,8f30fbf2-4476-4f09-97e7-e74d257b2020,Aedes geniculatus,mosquito,AT,46.6033,14.31730,exact_coords,841e163ffffffff,2022,7,AT211,50,Ovitrap,7
120155,a2fc7a3f-6c4b-407b-a3de-498a47984795,Aedes geniculatus,mosquito,AT,48.3372,14.31220,exact_coords,841e335ffffffff,2022,7,AT312,760,Ovitrap,91
120156,03fe780c-1d57-4c83-b444-fcdb0d03e25b,Culex,mosquito,MD,46.9730,28.88900,exact_coords,841e51bffffffff,2021,8,MDG241175,98,Ovitrap,565
120157,9f187db5-18d7-49c3-82af-3e98cd8bc30d,Culex,mosquito,MD,46.9380,28.92800,exact_coords,841e51bffffffff,2020,10,MDG241175,48,Ovitrap,515
